In [ ]:
import json
import re
from pathlib import Path
from urllib.request import urlopen

import geopandas as gpd
import pandas as pd

In [ ]:
folder = Path.cwd()

with (folder / "data.json").open(encoding="utf-8-sig") as f:
    raw = json.load(f)

# 128-dim embeddings are ~3.6 MB of the 8.2 MB file and the map never uses them.
df = pd.DataFrame([{k: v for k, v in r.items() if k != "learned_representation"}
                   for r in raw])

# Cache the boundaries so reruns work without internet access.
cache = folder / "ne_50m_admin_0_countries.geojson"
if not cache.exists():
    url = ("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
           "master/geojson/ne_50m_admin_0_countries.geojson")
    with urlopen(url, timeout=60) as response:
        cache.write_bytes(response.read())

boundaries = json.loads(cache.read_text(encoding="utf-8"))
world = gpd.GeoDataFrame.from_features(boundaries["features"], crs="EPSG:4326")
world = world[["ADMIN", "geometry"]].rename(columns={"ADMIN": "Country"})
world = world[world["Country"] != "Antarctica"].copy()
COUNTRIES = set(world["Country"])

# --- resolving `location` ----------------------------------------------------
# The new file's `location` is free text: 446 distinct strings mixing countries,
# cities, regions and non-places. These tables turn it into a country.

US = "United States of America"
ALIAS = {
    "united states": US, "usa": US, "us": US, "u.s.": US, "u.s.a.": US,
    "america": US, "american": US, "washington, d.c.": US, "washington d.c.": US,
    "capitol": US,
    "uk": "United Kingdom", "britain": "United Kingdom",
    "british": "United Kingdom", "england": "United Kingdom",
    "palestinian territories": "Palestine", "gaza": "Palestine",
    "gaza strip": "Palestine", "west bank": "Palestine", "hebron": "Palestine",
    "jenin": "Palestine",
    "israeli": "Israel", "chinese": "China", "mainland china": "China",
    "brazilian": "Brazil", "australian": "Australia", "mexican": "Mexico",
    "ukrainian": "Ukraine", "russian": "Russia", "german": "Germany",
    "french": "France", "indian": "India", "japanese": "Japan",
    "korean": "South Korea", "filipino": "Philippines",
    "czech republic": "Czechia", "serbia": "Republic of Serbia",
    "tanzania": "United Republic of Tanzania",
}

US_STATES = {
    "alabama", "alaska", "arizona", "arkansas", "california", "colorado",
    "connecticut", "delaware", "florida", "georgia", "hawaii", "idaho",
    "illinois", "indiana", "iowa", "kansas", "kentucky", "louisiana", "maine",
    "maryland", "massachusetts", "michigan", "minnesota", "mississippi",
    "missouri", "montana", "nebraska", "nevada", "new hampshire", "new jersey",
    "new mexico", "new york", "north carolina", "north dakota", "ohio",
    "oklahoma", "oregon", "pennsylvania", "rhode island", "south carolina",
    "south dakota", "tennessee", "texas", "utah", "vermont", "virginia",
    "washington", "west virginia", "wisconsin", "wyoming",
}

# Strings that name no place at all -> treated like the old file's "Global".
NON_PLACE = re.compile(
    r"\b(online|internet|web|worldwide|global|social media|platform|"
    r"facebook|youtube|twitter|tiktok|discord|telegram|twitch|reddit|"
    r"4chan|8kun|gab|tumblr|spotify|github|dark web|not specified|"
    r"home environment)\b", re.I)

_lower = {c.lower(): c for c in COUNTRIES}


def resolve(text):
    """Free-text location -> a Natural Earth country name, or 'Global'."""
    s = " ".join(str(text).split()).strip(" .,")
    if not s:
        return "Global"
    low = s.lower()

    # 1. the whole string is already a country (or a known alias for one)
    if low in _lower:
        return _lower[low]
    if low in ALIAS:
        return ALIAS[low]

    # 2. "City, Country" / "Region, Country" -> take the part after the comma
    parts = [p.strip() for p in s.split(",") if p.strip()]
    for part in reversed(parts):
        p = part.lower()
        if p in _lower:
            return _lower[p]
        if p in ALIAS:
            return ALIAS[p]

    # 3. a country, demonym or US state appears somewhere inside the string.
    #    Earliest mention wins, so "Texas-Mexico border" reads as the US side
    #    and "Kenya and Nepal" as Kenya.
    hits = []
    for name, country in list(_lower.items()) + list(ALIAS.items()):
        if len(name) < 4:
            continue
        m = re.search(r"\b" + re.escape(name) + r"\b", low)
        if m:
            hits.append((m.start(), country))
    for state in US_STATES:
        m = re.search(r"\b" + re.escape(state) + r"\b", low)
        if m:
            hits.append((m.start(), US))
    # short forms that the length guard above skips
    for pat, country in ((r"\bu\.?s\.?a?\b", US), (r"\bu\.?k\.?\b", "United Kingdom"),
                         (r"\bd\.?c\.?\b", US), (r"\bca\b", US), (r"\bny\b", US)):
        m = re.search(pat, low)
        if m:
            hits.append((m.start(), country))
    if hits:
        return min(hits)[1]

    # 5. no place at all (online, worldwide, unspecified)
    if NON_PLACE.search(low):
        return "Global"
    return "Global"


df["map_country"] = df["location"].map(resolve)

# --- harm taxonomy -----------------------------------------------------------
# harm_category is a LIST: 998 reports carry one label, 158 two, 3 three.
# A report is kept once and tagged with all of its labels, so filtering by any
# class shows every report that carries it.
def labels(cell):
    out = []
    for h in cell or []:
        sub = h.get("subcategory") or "Unclassified"
        sid = h.get("subcategory_id") or ""
        out.append({
            "label": (f"{sid}  {sub}".strip()),
            "branch": (h.get("branch") or "Unclassified").replace(
                "UNMAPPED", "Unclassified"),
            "event": h.get("event_type") or "Unspecified",
        })
    return out or [{"label": "Unclassified", "branch": "Unclassified",
                    "event": "Unspecified"}]


df["harms"] = df["harm_category"].map(labels)
df["harm_labels"] = df["harms"].map(lambda hs: [h["label"] for h in hs])
df["harm_branches"] = df["harms"].map(lambda hs: sorted({h["branch"] for h in hs}))
# Acute / chronic lives INSIDE harm_category, not in the top-level event_type
# (which is a free-text incident description with 1,044 distinct values).
df["event_kind"] = df["harms"].map(
    lambda hs: ", ".join(sorted({h["event"] for h in hs})))

df["text"] = df["original_evidence_span"].fillna("").astype(str)
df["url"] = df["source"].fillna("").astype(str)
df["date"] = df["event_date"].fillna("").astype(str)

mapped = df["map_country"].isin(COUNTRIES)

print(f"Records in predicted.json: {len(df):,}")
print(f"Placed on the map:         {mapped.sum():,} "
      f"({100 * mapped.mean():.0f}%)")
print(f"No usable location:        {(~mapped).sum():,}")
print(f"Countries covered:         {df.loc[mapped, 'map_country'].nunique()}")
print(f"Harm labels total:         {df['harm_labels'].map(len).sum():,} "
      f"across {len(df):,} reports")


In [ ]:
import json as _json
import re
from collections import Counter

import folium
import numpy as np
from branca.element import Element
from shapely.geometry import Point
from shapely.prepared import prep

OCEAN = "#0c0c10"
LAND = "#3a3a3a"
PALETTE = ["#4C78A8", "#F58518", "#E45756", "#54A24B", "#B279A2", "#7F7F7F"]

placed = df.loc[mapped]

# A report can carry several harm labels, so class counts are "reports
# mentioning this class" and add up to more than the report total.
branch_hits = Counter(b for bs in placed["harm_branches"] for b in bs)
branches = [b for b, _ in branch_hits.most_common()]
branch_color = {b: PALETTE[i % len(PALETTE)] for i, b in enumerate(branches)}
branch_color["Unclassified"] = "#9C9C9C"

label_hits = Counter(l for ls in placed["harm_labels"] for l in ls)
label_branch = {}
for hs in placed["harms"]:
    for h in hs:
        label_branch[h["label"]] = h["branch"]

totals = placed.groupby("map_country").size()

# --- one dot per report ------------------------------------------------------
# predicted.json has no coordinates, only a location string, so each dot is
# placed at random inside its country. Seeded, so the layout never shifts.
_rng = np.random.default_rng(20260907)
_geom = dict(zip(world["Country"], world.geometry))


def scatter(country, n):
    poly = _geom[country]
    inside = prep(poly)
    minx, miny, maxx, maxy = poly.bounds
    pts = []
    while len(pts) < n:
        k = max(4 * (n - len(pts)), 32)
        for x, y in zip(_rng.uniform(minx, maxx, k), _rng.uniform(miny, maxy, k)):
            if inside.contains(Point(x, y)):
                pts.append((round(y, 4), round(x, 4)))
                if len(pts) == n:
                    break
    return pts


def make_title(url, text):
    """Headline from the URL slug; falls back to the first sentence."""
    slug = re.sub(r"\.(html?|php|aspx?)$", "", url.rstrip("/").rsplit("/", 1)[-1])
    slug = re.sub(r"[-_]+", " ", slug).strip()
    if len(slug.split()) >= 3:
        return slug[:1].upper() + slug[1:]
    first = text.strip().split(". ")[0]
    return first[:90] + "…" if len(first) > 90 else first


records = []
for _country, _grp in placed.groupby("map_country"):
    for (_lat, _lon), (_, r) in zip(scatter(_country, len(_grp)), _grp.iterrows()):
        # colour by the report's first (primary) harm class
        primary = r["harms"][0]["branch"]
        records.append({
            "y": _lat, "x": _lon,
            "c": branch_color[primary],
            "b": r["harm_branches"],
            "h": r["harm_labels"],
            "t": make_title(r["url"], r["text"]),
            "s": r["text"], "u": r["url"], "d": r["date"], "k": _country,
            "e": r["event_kind"],
        })
print(f"Dots placed: {len(records):,} across {totals.size} countries")

# --- map ---------------------------------------------------------------------
# No tile layer: the land is drawn from the boundary polygons, stroked in its
# own fill colour so the continents read as one mass with no borders.
harm_map = folium.Map(location=[20, 0], zoom_start=2, tiles=None,
                      prefer_canvas=True)
folium.GeoJson(
    _json.loads(world[["Country", "geometry"]].to_json()),
    style_function=lambda f: {"fillColor": LAND, "color": LAND,
                              "weight": 1, "fillOpacity": 1},
    interactive=False,          # clicks belong to the dots, not the land
).add_to(harm_map)
harm_map.fit_bounds([[-55, -180], [80, 180]])

legend_rows, filters = "", []
for b in branches:
    filters.append({"k": "b", "v": b, "c": branch_color[b]})
    legend_rows += (
        '<div class="k-row" data-i="%d"><i style="background:%s"></i>%s'
        '<span class="k-n">%d</span></div>'
        % (len(filters) - 1, branch_color[b], b, branch_hits[b])
    )
    subs = sorted(((l, n) for l, n in label_hits.items()
                   if label_branch[l] == b), key=lambda x: -x[1])
    for lab, n in subs:
        filters.append({"k": "s", "v": lab, "c": branch_color[b]})
        legend_rows += (
            '<div class="k-sub" data-i="%d">%s<span class="k-n">%d</span></div>'
            % (len(filters) - 1, lab, n)
        )

PANEL = """
<style>
.folium-map { background: __OCEAN__; }
.hkey { position:absolute; right:12px; bottom:22px; z-index:9999; width:320px;
  background:rgba(255,255,255,.95); border-radius:6px; padding:10px 12px;
  box-shadow:0 0 12px rgba(0,0,0,.4); color:#222;
  font:12px/1.45 Arial,Helvetica,sans-serif;
  max-height:calc(100vh - 44px); overflow-y:auto; }
.hkey h4 { margin:0 0 2px; font-size:13px; }
.hkey .k-sub2 { color:#666; margin-bottom:7px; }
.hkey .k-scroll { max-height:52vh; overflow-y:auto; }
.hkey .k-row { display:flex; align-items:center; gap:7px; margin-top:6px;
  font-weight:bold; line-height:1.35; }
.hkey .k-row i { width:11px; height:11px; border-radius:50%; display:inline-block;
  flex:0 0 auto; }
.hkey .k-sub { display:flex; gap:7px; padding-left:18px; color:#444;
  font-size:11.5px; line-height:1.35; }
.hkey .k-n { margin-left:auto; color:#555; font-weight:normal;
  white-space:nowrap; }
.hkey .k-row, .hkey .k-sub { cursor:pointer; border-radius:3px;
  padding:1px 4px; margin-left:-4px; }
.hkey .k-row:hover, .hkey .k-sub:hover { background:#e9ecf3; }
.hkey .k-row.on, .hkey .k-sub.on { background:#dce6f7; }
.hkey .k-all { width:100%; margin:2px 0 8px; padding:5px 8px; cursor:pointer;
  border:1px solid #c3c3cc; border-radius:4px; background:#f2f2f5;
  font:bold 11.5px Arial,Helvetica,sans-serif; color:#333; }
.hkey .k-all:hover { background:#e6e6ec; }
.hkey .k-all.on { background:#dce6f7; border-color:#9db6de; }
.hkey .k-note { margin-top:9px; padding-top:7px; border-top:1px solid #ddd;
  color:#666; font-style:italic; font-size:11px; }
.hbar { position:absolute; left:0; right:0; bottom:0; z-index:9998; display:none;
  background:rgba(247,247,249,.975); border-top:1px solid #c9c9cf;
  padding:13px 46px 15px; text-align:center; max-height:34vh; overflow-y:auto;
  font:13px/1.55 Arial,Helvetica,sans-serif; }
.hbar .b-x { position:absolute; right:16px; top:9px; cursor:pointer;
  font-size:21px; line-height:1; color:#666; }
.hbar .b-x:hover { color:#000; }
.hbar .b-t { font-weight:bold; font-size:14px; margin-bottom:5px; }
.hbar .b-h { font-weight:normal; color:#a3213a; }
.hbar .b-s { color:#222; max-width:1050px; margin:0 auto 7px; }
.hbar .b-m { color:#555; font-size:12px; }
.hbar .b-m a { color:#1a5fb4; }
.hbar .b-list { max-width:900px; margin:0 auto; text-align:left; }
.hbar .b-pick { display:block; width:100%; background:none; border:0;
  border-top:1px solid #e3e3e8; padding:6px 2px; cursor:pointer;
  font:13px Arial,Helvetica,sans-serif; color:#1a5fb4; text-align:left; }
.hbar .b-pick:hover { background:#ecedf2; }
.hbar .b-pick .b-h { color:#a3213a; }
.hbar .b-more { background:none; border:0; margin-top:6px; cursor:pointer;
  font:12px Arial,Helvetica,sans-serif; color:#1a5fb4; text-decoration:underline; }
</style>
<div class="hbar" id="hbar"></div>
<div class="hkey">
  <h4>AI harm reports</h4>
  <div class="k-sub2" id="k-count">__N__ reports &middot; one dot each</div>
  <button class="k-all on" id="k-all">Show all reports</button>
  <div class="k-scroll">__ROWS__</div>
  <div class="k-note">Each dot is one report, coloured by its harm class and
  scattered at random inside its country: dots show which country a report
  concerns, <b>not</b> where the incident happened. Click a dot to read it.
  Some reports carry two or three classes, so the class counts add up to more
  than the report total.</div>
</div>
<script>
window.addEventListener('load', function () {
  var map = __MAP__;
  var RECORDS = __RECORDS__, FILTERS = __FILTERS__;
  var TOTAL = RECORDS.length;
  var bar = document.getElementById('hbar');
  var key = document.querySelector('.hkey');
  var count = document.getElementById('k-count');
  var allBtn = document.getElementById('k-all');
  var rows = document.querySelectorAll('.hkey [data-i]');
  var active = null, shown = RECORDS;
  var dots = L.layerGroup().addTo(map);

  RECORDS.forEach(function (r) {
    r.m = L.circleMarker([r.y, r.x], {
      radius: 4, weight: 0.7, color: '#141418', opacity: 0.9,
      fillColor: r.c, fillOpacity: 0.92
    }).bindTooltip(r.t, { direction: 'top', opacity: 0.95 });
  });

  function close_() {
    bar.style.display = 'none';
    key.style.bottom = '';
    key.style.maxHeight = '';
  }

  function open_(html) {
    bar.style.display = 'block';
    bar.innerHTML = '<span class="b-x">&times;</span>' + html;
    key.style.bottom = (bar.offsetHeight + 14) + 'px';
    key.style.maxHeight = (window.innerHeight - bar.offsetHeight - 40) + 'px';
    bar.querySelector('.b-x').onclick = close_;
  }

  function report(r, others) {
    open_('<div class="b-t">' + r.t + ' <span class="b-h">| Harm: '
      + r.h.join(' &middot; ') + '</span></div>'
      + '<div class="b-s">' + r.s + '</div>'
      + '<div class="b-m">' + r.k + ' &middot; ' + r.d + ' &middot; ' + r.e
      + ' &middot; Source: <a href="' + r.u + '" target="_blank"'
      + ' rel="noopener">' + r.u.split('/')[2] + '</a></div>'
      + (others && others.length
          ? '<button class="b-more">' + others.length + ' more report'
            + (others.length > 1 ? 's' : '') + ' at this spot &#9662;</button>'
          : ''));
    var more = bar.querySelector('.b-more');
    if (more) { more.onclick = function () { picker([r].concat(others)); }; }
  }

  function picker(rs) {
    open_('<div class="b-t">' + rs.length + ' reports here</div>'
      + '<div class="b-list">' + rs.map(function (rr, i) {
        return '<button class="b-pick" data-i="' + i + '">' + rr.t
          + ' <span class="b-h">| ' + rr.h.join(' &middot; ')
          + '</span></button>';
      }).join('') + '</div>');
    Array.prototype.forEach.call(bar.querySelectorAll('.b-pick'), function (b) {
      b.onclick = function () {
        var i = +b.dataset.i;
        report(rs[i], rs.filter(function (_, j) { return j !== i; }));
      };
    });
  }

  // Dots overlap heavily, so a per-marker click would leave the ones
  // underneath unreachable. Resolve every click to the dots within 10px.
  map.on('click', function (e) {
    var p = map.latLngToContainerPoint(e.latlng);
    var hits = [];
    shown.forEach(function (r) {
      var d = p.distanceTo(map.latLngToContainerPoint(L.latLng(r.y, r.x)));
      if (d <= 10) { hits.push({ r: r, d: d }); }
    });
    if (!hits.length) { return; }
    hits.sort(function (a, b) { return a.d - b.d; });
    report(hits[0].r, hits.slice(1).map(function (hh) { return hh.r; }));
  });

  function draw(list, label) {
    shown = list;
    dots.clearLayers();
    // In a filtered view every dot is shown in the selected class's colour.
    // A report can carry several classes, so otherwise a dot kept by its
    // secondary class would still wear its primary colour and look wrong.
    var fc = active === null ? null : FILTERS[active].c;
    list.forEach(function (r) {
      r.m.setStyle({ fillColor: fc || r.c });
      dots.addLayer(r.m);
    });
    count.innerHTML = label;
    close_();
    Array.prototype.forEach.call(rows, function (el) {
      el.classList.toggle('on', String(active) === el.dataset.i);
    });
    allBtn.classList.toggle('on', active === null);
  }

  function showAll() {
    active = null;
    draw(RECORDS, TOTAL.toLocaleString() + ' reports &middot; one dot each');
  }

  Array.prototype.forEach.call(rows, function (el) {
    el.onclick = function () {
      if (String(active) === el.dataset.i) { showAll(); return; }
      active = el.dataset.i;
      var f = FILTERS[active];
      var list = RECORDS.filter(function (r) {
        return (f.k === 'b' ? r.b : r.h).indexOf(f.v) >= 0;
      });
      draw(list, list.length.toLocaleString() + ' of '
        + TOTAL.toLocaleString() + ' reports &middot; ' + f.v);
    };
  });
  allBtn.onclick = showAll;
  showAll();
});
</script>
"""

panel = (
    PANEL.replace("__OCEAN__", OCEAN)
    .replace("__ROWS__", legend_rows)
    .replace("__N__", "{:,}".format(len(records)))
    .replace("__RECORDS__", _json.dumps(records, ensure_ascii=False))
    .replace("__FILTERS__", _json.dumps(filters, ensure_ascii=False))
    .replace("__MAP__", harm_map.get_name())
)
harm_map.get_root().html.add_child(Element(panel))
harm_map.save(str(folder / "map.html"))

# Display inside Jupyter.
display(harm_map)